In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, lit, expr, round, col

spark = SparkSession.builder.getOrCreate()

SILVER_TABLE = "sales_tax_catalog.silver.sales_tax_clean"
GOLD_TABLE = "sales_tax_catalog.gold.sales_tax_rag_context"

def aggregate_gold():
    df_clean = spark.table(SILVER_TABLE)
    
    # A Mágica do Data-to-Text: Criando frases descritivas para o Vector Search
    df_gold = df_clean.withColumn(
        "document_text",
        concat_ws(" ",
            lit("In month"), col("report_month"),
            lit("of year"), col("report_year"),
            lit("("), col("report_period_type"), lit("), the city of"), col("city_name"),
            lit("received a net sales tax payment of $"), 
            round(col("net_payment"), 2), 
            lit("and the total payment to date is $"), 
            round(col("payment_to_date"), 2), lit(".")
        )
    ).withColumn("id_record", expr("uuid()")) # UUID obrigatório para o Databricks Vector Search
    
    # Gravando na Gold mantendo o Change Data Feed ativado
    df_gold.select("id_record", "city_name", "document_text") \
        .write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable(GOLD_TABLE)
        
    print(f"Camada Gold RAG gerada com sucesso: {GOLD_TABLE}")

if __name__ == "__main__":
    aggregate_gold()